# Integrated Gradients cross-check of the OLL screen

Our out-of-lung-localization (OLL) screen uses Grad-CAM. This checks whether the finding is **method-specific** by recomputing OLL with a second attribution method, Integrated Gradients (IG), and comparing per-disease. If the two agree, the localization result isn't a Grad-CAM artifact (cf. Adebayo et al., sanity checks for saliency).

**Both methods run on the SAME GPU here**, so the comparison isolates method, not device (we already know OLL is device-sensitive, so we hold device fixed).

**Before running:** `Runtime → Change runtime type → GPU`.

**You need** `chexpert_data.zip` from Drive (images are gitignored). IG is ~`n_steps`× the cost of Grad-CAM, which is why this runs on GPU.

**Send back:** `oll_crosscheck.zip` (the two OLL CSVs: Grad-CAM and IG).

### 1. Clone + install (captum is the new dep)

In [1]:
!git clone -b feat/extra-experiments https://github.com/su-andrew/cs229-shortcut-detection.git
%cd cs229-shortcut-detection
!pip install -q torch torchvision torchxrayvision grad-cam scikit-image scikit-learn pandas numpy matplotlib pyyaml captum

Cloning into 'cs229-shortcut-detection'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 279 (delta 10), reused 14 (delta 6), pack-reused 238 (from 1)
Receiving objects: 100% (279/279), 51.45 MiB | 23.93 MiB/s, done.
Resolving deltas: 100% (154/154), done.
/content/cs229-shortcut-detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 64.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 31.9 MB/s eta 0:00:00


### 2. Confirm GPU

In [2]:
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


### 3. Mount Drive + unzip data
Edit `DATA_ZIP` to its path in your Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS to your data zip's Drive path >>>
DATA_ZIP = '/content/drive/MyDrive/chexpert_data.zip'

!unzip -q "$DATA_ZIP" -d data/
import os
need = ['data/chexpert/PNG_valid', 'data/chexpert/metadata.csv',
        'results/predictions_val.csv']
# predictions_val.csv ships in the repo? if not, regenerate the baseline first.
missing = [p for p in need if not os.path.exists(p)]
print('missing:', missing if missing else 'none')
assert os.path.exists('data/chexpert/PNG_valid'), 'data not unzipped — check DATA_ZIP'

Mounted at /content/drive
missing: ['results/predictions_val.csv']


### 4. Ensure baseline predictions exist
OLL needs `results/predictions_val.csv`. If it's not in the repo clone, regenerate it (cheap, inference-only).

In [4]:
import os
if not os.path.exists('results/predictions_val.csv'):
    !python -m src.baseline --device cuda
print('predictions ready:', os.path.exists('results/predictions_val.csv'))

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
Running baseline over 234 validation images on cuda ...

Per-disease AUROC:
           label    auroc  n_pos  n_neg  n_excluded  published_ref_approx
     Atelectasis      NaN     35      0         199                  0.85
    Cardiomegaly 0.850877     19     24         191                  0.83
   Consolidation 0.857778     15     45         174                  0.90
           Edema 0.860248     46     21         167                  0.93
Pleural Effusion 0.876610     63     53         118                  0.93

Wrote results/auroc.csv
Wrote results/predictions_val.csv  (234 rows)
predictions ready: True


### 5. OLL with Grad-CAM and with IG, all 4 labels, same GPU

In [5]:
!python -m src.ola --num-labels 4 --device cuda --method gradcam
!python -m src.ola --num-labels 4 --device cuda --method ig

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth -O /root/.torchxrayvision/models_data/pspnet_chestxray_best_model_4.pth`
[██████████████████████████████████████████████████]
           label  oll_pos_mean  oll_pos_lo  oll_pos_hi  oll_neg_mean  oll_fp_mean  oll_fn_mean  n_pos  n_neg  n_empty_mask  n_boot  ranked
   Consolidation      0.655468    0.605015    0.704760      0.769219     0.756829          NaN     13     34             0    2000    True
           Edema      0.637360    0.593679    0.683336      0.772862     0.730326     0.934011     44     19             0    2000    True
Pleural Effusion      0.550967    0.496641    0.606188      0.675845     0.610346     0.692841     57     40             0    2000    True
    Cardiomegaly      0.492219    0.427245    0.562493      0.714516     0.603970     0.741917     17     19             0    2000    True
Wrote results/oll_by_disease.csv
Wrote results/figu

### 6. Compare — does IG agree with Grad-CAM?

In [6]:
import pandas as pd
g = pd.read_csv('results/oll_by_disease.csv').set_index('label')['oll_pos_mean']
i = pd.read_csv('results/oll_by_disease_ig.csv').set_index('label')['oll_pos_mean']
print(f'{"label":18s} GradCAM   IG     (diff)')
for l in g.index:
    if l in i.index:
        print(f'  {l:18s} {g[l]:.3f}   {i[l]:.3f}  ({i[l]-g[l]:+.3f})')
# rank agreement is the headline: do both methods order the diseases the same?
print('\nGrad-CAM ranking:', list(g.sort_values(ascending=False).index))
print('IG ranking:      ', list(i.sort_values(ascending=False).index))

label              GradCAM   IG     (diff)
  Consolidation      0.655   0.579  (-0.076)
  Edema              0.637   0.461  (-0.177)
  Pleural Effusion   0.551   0.650  (+0.099)
  Cardiomegaly       0.492   0.524  (+0.032)

Grad-CAM ranking: ['Consolidation', 'Edema', 'Pleural Effusion', 'Cardiomegaly']
IG ranking:       ['Pleural Effusion', 'Consolidation', 'Cardiomegaly', 'Edema']


### 7. Zip the two OLL CSVs + download → send to Jonathan

In [7]:
!cd results && zip -q /content/oll_crosscheck.zip oll_by_disease.csv oll_by_disease_ig.csv && echo done
from google.colab import files
files.download('/content/oll_crosscheck.zip')

done


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>